# 1.3 Input Gathering and Architecture Smoke Test

Purpose:
- Create a structured assessment input
- Run placeholder face, speech, and metadata modules
- Fuse module outputs
- Generate a triage report

This proves that the project architecture is implemented end-to-end.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.capture.schemas import PatientMetadata
from rural_stroke_assist.capture.input_handler import create_assessment_input
from rural_stroke_assist.modules.face_module import analyze_face_image
from rural_stroke_assist.modules.speech_module import analyze_speech_audio
from rural_stroke_assist.modules.metadata_module import analyze_metadata
from rural_stroke_assist.fusion.fusion_engine import fuse_results
from rural_stroke_assist.reporting.report_generator import generate_text_report
from rural_stroke_assist.config import FACE_DATA_DIR

In [2]:
from rural_stroke_assist.preprocessing.image import list_image_files

image_paths = list_image_files(FACE_DATA_DIR)
sample_image_path = image_paths[0]

sample_image_path

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/data/raw/face_stroke_images/Annotated stroke and non stroke Dataset/NonStroke/img_0022.jpg')

In [3]:
metadata = PatientMetadata(
    age=68,
    sex="male",
    symptom_onset_minutes=120,
    hypertension=True,
    diabetes=False,
    previous_stroke=False,
    notes="Demo assessment for milestone smoke test.",
)

metadata

PatientMetadata(age=68, sex='male', symptom_onset_minutes=120, hypertension=True, diabetes=False, previous_stroke=False, notes='Demo assessment for milestone smoke test.')

In [4]:
assessment_input = create_assessment_input(
    metadata=metadata,
    face_image_path=sample_image_path,
    speech_audio_path=None,
)

assessment_input

AssessmentInput(session_id='a208ff15-0aba-4fbe-aa15-a6f254271e53', face_video_path=None, speech_audio_path=None, face_image_path=WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/data/raw/face_stroke_images/Annotated stroke and non stroke Dataset/NonStroke/img_0022.jpg'), metadata=PatientMetadata(age=68, sex='male', symptom_onset_minutes=120, hypertension=True, diabetes=False, previous_stroke=False, notes='Demo assessment for milestone smoke test.'))

In [5]:
face_result = analyze_face_image(assessment_input.face_image_path)
face_result

FaceModuleResult(facial_asymmetry_score=0.5, confidence=0.5, evidence=['Placeholder face module result.'], warnings=['Real facial landmark analysis not implemented yet.'])

In [6]:
speech_result = analyze_speech_audio(assessment_input.speech_audio_path)
speech_result

SpeechModuleResult(speech_abnormality_score=0.0, confidence=0.0, evidence=[], warnings=['No speech audio provided.'])

In [7]:
metadata_result = analyze_metadata(assessment_input.metadata)
metadata_result

MetadataModuleResult(contextual_risk_score=0.6, confidence=0.8, evidence=['Age is above 60.', 'Hypertension reported.', 'Symptoms started within an urgent time window.'], warnings=[])

In [8]:
fusion_result = fuse_results(
    face_result=face_result,
    speech_result=speech_result,
    metadata_result=metadata_result,
)

fusion_result

FusionResult(final_risk_score=0.345, triage_level='Low concern', confidence=0.385, evidence=['Placeholder face module result.', 'Age is above 60.', 'Hypertension reported.', 'Symptoms started within an urgent time window.'], warnings=['Real facial landmark analysis not implemented yet.', 'No speech audio provided.'])

In [9]:
report = generate_text_report(fusion_result)
print(report)

RuralStroke-Assist Triage Report

Triage Level:
Low concern

Final Risk Score:
0.345

System Confidence:
0.385

Evidence:
- Placeholder face module result.
- Age is above 60.
- Hypertension reported.
- Symptoms started within an urgent time window.

Warnings:
- Real facial landmark analysis not implemented yet.
- No speech audio provided.

Important Safety Note:
This system does not diagnose stroke. It provides decision-support guidance for early triage and escalation.


In [10]:
from rural_stroke_assist.config import SMOKE_TEST_REPORT_PATH

SMOKE_TEST_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

SMOKE_TEST_REPORT_PATH.write_text(report, encoding="utf-8")

print(f"Saved report to: {SMOKE_TEST_REPORT_PATH}")

Saved report to: C:\Users\Asus\Downloads\Uni_work\Grad-Project\rural-stroke-assist\reports\milestone_1_smoke_test_report.txt
